In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
import os

# Facial Landmarks #

In [2]:
# Load the DataFrame first
df_fl = pd.read_csv('Facial_4HZ/labeled_combined_facial_features_4HZ.csv')

# Dynamically select columns that contain 
selected_columns = [col for col in df_fl.columns if 'X' in col or 'Y' in col]

# Add 'user', 'session', and 'label' columns to the selected columns
selected_columns += ['user', 'session', 'label']

# Load the relevant columns based on the selection
df_fl = pd.read_csv('Facial_4HZ/labeled_combined_facial_features_4HZ.csv',
                      usecols=selected_columns)


### Interpolate Missing Values in Facial Landmarks Data ###

In [3]:
# Select only numerical columns for interpolation
numerical_columns = df_fl.select_dtypes(include=['number']).columns

# Group the data by 'user' and 'session' and apply interpolation separately for each group
df_fl[numerical_columns] = df_fl.groupby(['user', 'session'])[numerical_columns].apply(
    lambda group: group.interpolate(method='linear', axis=0, limit_direction='forward')
).reset_index(drop=True, level=['user', 'session'])

In [6]:
# Create necessary directories if they don't exist
os.makedirs("4HZ_Individual_Data", exist_ok=True)

# Optionally save the clean data to a new CSV file
df_fl.to_csv('4HZ_Individual_Data/final_FL_4HZ.csv', index=False)

In [ ]:
df_fl['DrowsinessLevel'] = df_fl['label'].apply(lambda x: 0 if x < 4 else 1 if x<7 else 2)
df_fl = df_fl.drop(columns=['label'])

### Applying SVM for classification (Facial Landmarks) ###

In [ ]:
%%time
# --- Setup Cross-validation ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Separate features and target
X = df_fl.drop(columns=['DrowsinessLevel', 'session', 'user'])
y = df_fl['DrowsinessLevel']

# Apply Min-Max normalization
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X)

# --- PCA Feature Selection ---
# PCA to retain a specific amount of variance
pca = PCA(n_components=0.90) 
X_pca = pca.fit_transform(X_normalized)

# Print the number of components selected and the explained variance ratio
print(f"Number of principal components selected: {X_pca.shape[1]}")
print(f"Explained variance ratio per component: {pca.explained_variance_ratio_}")
print(f"Cumulative explained variance: {np.cumsum(pca.explained_variance_ratio_)}")

# --- SVM ---
svm_clf = SVC(kernel='rbf', C=0.1, gamma='auto', cache_size=200,class_weight='balanced', random_state=42) 
svm_cv_scores = cross_val_score(svm_clf, X_pca, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("SVM CV Accuracy: {:.2f} (+/- {:.2f})".format(svm_cv_scores.mean(), svm_cv_scores.std() * 2))
y_cv_pred = cross_val_predict(svm_clf, X_pca, y, cv=cv, n_jobs=-1)
print("SVM CV Classification Report:\n", classification_report(y, y_cv_pred, digits=4))
cm = confusion_matrix(y, y_cv_pred, normalize='true')
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y)).plot(cmap='Blues', values_format=".2f")
plt.title("SVM Normalized Confusion Matrix (5-Fold CV)")
plt.show()

# --- Random Forest ---
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)
rf_cv_scores = cross_val_score(rf_clf, X_pca, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("RF CV Accuracy: {:.2f} (+/- {:.2f})".format(rf_cv_scores.mean(), rf_cv_scores.std() * 2))
y_cv_pred = cross_val_predict(rf_clf, X_pca, y, cv=cv, n_jobs=-1)
print("RF CV Classification Report:\n", classification_report(y, y_cv_pred, digits=4))
cm = confusion_matrix(y, y_cv_pred, normalize='true')
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y)).plot(cmap='Blues', values_format=".2f")
plt.title("Random Forest Normalized Confusion Matrix (5-Fold CV)")
plt.show()


# Facial Action Units  #

In [9]:
# Load the DataFrame first
df_fau = pd.read_csv('Facial_4HZ/labeled_combined_facial_features_4HZ.csv')

# Identify columns that contain 'X' or 'Y' in their names
columns_to_drop = [col for col in df_fau.columns if 'X' in col or 'Y' in col]
columns_to_drop += ['Frame', 'Frame_Number']

# Drop the identified columns
df_fau = df_fau.drop(columns=columns_to_drop)

In [10]:
# Replace infinite values with NaN
df_fau.replace([np.inf, -np.inf], np.nan, inplace=True)

### Interpolate Missing Values in FAU Data ###

In [11]:
# Select only numerical columns for interpolation
numerical_columns = df_fau.select_dtypes(include=['number']).columns

# Group the data by 'user' and 'session' and apply interpolation separately for each group
df_fau[numerical_columns] = df_fau.groupby(['user', 'session'])[numerical_columns].apply(
    lambda group: group.interpolate(method='linear', axis=0, limit_direction='forward')
).reset_index(drop=True, level=['user', 'session'])

In [12]:
# Optionally save the clean data to a new CSV file
df_fau.to_csv('4HZ_Individual_Data/final_FAU_4HZ.csv', index=False)

In [13]:
df_fau['DrowsinessLevel'] = df_fau['label'].apply(lambda x: 0 if x < 4 else 1 if x<7 else 2)
df_fau = df_fau.drop(columns=['label'])

### Applying SVM for classification (FAU) ###

In [ ]:
%%time
# --- Setup Cross-validation ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Separate features and target
X = df_fau.drop(columns=['DrowsinessLevel', 'session', 'user'])
y = df_fau['DrowsinessLevel']

# Apply Min-Max normalization
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X)

# --- PCA Feature Selection ---
# PCA to retain a specific amount of variance
pca = PCA(n_components=0.90) 
X_pca = pca.fit_transform(X_normalized)

# Print the number of components selected and the explained variance ratio
print(f"Number of principal components selected: {X_pca.shape[1]}")
print(f"Explained variance ratio per component: {pca.explained_variance_ratio_}")
print(f"Cumulative explained variance: {np.cumsum(pca.explained_variance_ratio_)}")

# --- SVM ---
svm_clf = SVC(kernel='rbf', C=0.1, gamma='auto', cache_size=200,class_weight='balanced', random_state=42) 
svm_cv_scores = cross_val_score(svm_clf, X_pca, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("SVM CV Accuracy: {:.2f} (+/- {:.2f})".format(svm_cv_scores.mean(), svm_cv_scores.std() * 2))
y_cv_pred = cross_val_predict(svm_clf, X_pca, y, cv=cv, n_jobs=-1)
print("SVM CV Classification Report:\n", classification_report(y, y_cv_pred, digits=4))
cm = confusion_matrix(y, y_cv_pred, normalize='true')
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y)).plot(cmap='Blues', values_format=".2f")
plt.title("SVM Normalized Confusion Matrix (5-Fold CV)")
plt.show()

# --- Random Forest ---
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)
rf_cv_scores = cross_val_score(rf_clf, X_pca, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("RF CV Accuracy: {:.2f} (+/- {:.2f})".format(rf_cv_scores.mean(), rf_cv_scores.std() * 2))
y_cv_pred = cross_val_predict(rf_clf, X_pca, y, cv=cv, n_jobs=-1)
print("RF CV Classification Report:\n", classification_report(y, y_cv_pred, digits=4))
cm = confusion_matrix(y, y_cv_pred, normalize='true')
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y)).plot(cmap='Blues', values_format=".2f")
plt.title("Random Forest Normalized Confusion Matrix (5-Fold CV)")
plt.show()
